In [2]:
#imports
from pathlib import Path
from rapidocr import RapidOCR
from rapidocr.utils.output import RapidOCROutput
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Helper Functions

In [5]:
from dataclasses import dataclass


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / ".git").exists():
            return folder
    raise FileNotFoundError(f"No git repo found above {start}")


REPO_ROOT = find_repo_root()

train_path = REPO_ROOT / "images" / "train"

@dataclass
class Sample:
    path: Path     
    img: np.ndarray  # BGR color


def load_images(folder: Path, limit: int | None = None) -> list[Sample]:
    samples = []

    for file_path in sorted(Path(folder).resolve().iterdir()):
        if not file_path.is_file():
            continue

        img = cv2.imread(str(file_path))
        if img is None or img.size == 0:
            continue

        try:
            rel_path = file_path.relative_to(REPO_ROOT)
        except ValueError:  # image lives outside the repo: keep the absolute path
            rel_path = file_path
        samples.append(Sample(rel_path, img))

        if limit is not None and len(samples) >= limit:
            break

    return samples

In [8]:
def visualize_results(results):
    for res in results:
        fig, row = plt.subplots(1, 2, figsize=(10, 10), layout="constrained",
                                gridspec_kw=dict(width_ratios=[1, 2]))  # vis() output is wider with teh results
        
        row[0].imshow(res["org_img"])
        row[0].set_title(res["name"])
        row[1].imshow(res["ocr_img"])
        row[1].set_title(f"RapidOCR - {res['name']}")

        row[0].set_axis_off()
        row[1].set_axis_off()
        plt.show()

In [9]:
samples = load_images(train_path, 3)

## 1 | RapidOCR

In [6]:
engine = RapidOCR()

def run_ocr(img: np.ndarray) -> RapidOCROutput | None:
    result = engine(img) 
    if not isinstance(result, RapidOCROutput) or result.boxes is None:
        return None
    return result

rapid_results = []

for sample in samples: 
    result = run_ocr(sample.img)
    org_img = cv2.cvtColor(sample.img, cv2.COLOR_BGR2RGB)  # cv2 loads BGR; matplotlib expects RGB

    if result is None: 
        continue

    vis_img = result.vis()  # BGR ndarray or None
    ocr_img = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB) if vis_img is not None else org_img
    rapid_results.append(dict(
        name=str(sample.path),
        text=result.txts or (), 
        scores=result.scores or (), 
        org_img=org_img,
        ocr_img=ocr_img, 
    ))

[INFO] 2026-11-14 19:13:32,295 [RapidOCR] base.py:23: Using engine_name: onnxruntime


[INFO] 2026-11-14 19:13:32,358 [RapidOCR] download_file.py:60: File exists and is valid: /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-14 19:13:32,359 [RapidOCR] main.py:63: Using /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-14 19:13:32,931 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-14 19:13:32,934 [RapidOCR] download_file.py:60: File exists and is valid: /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-14 19:13:32,935 [RapidOCR] main.py:63: Using /home/jmtcabili/Documents/DEV/TransacOCR/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-14 19:13:33,008 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-14 19:13:33,072 [RapidOCR] 

In [7]:
#For each pair of images and their results, I want a 1x2 subplot with the results printed below
visualize_results(rapid_results)

<Figure size 1000x1000 with 2 Axes>

<Figure size 1000x1000 with 2 Axes>

<Figure size 1000x1000 with 2 Axes>

## 2 | PyTesseract

In [8]:
import pandas as pd
import pytesseract
pd.set_option("display.max_colwidth", None)
tesseract_rows = []
for sample in samples:
    rgb = cv2.cvtColor(sample.img, cv2.COLOR_BGR2RGB)  # pytesseract reads ndarrays as RGB
    tesseract_rows.append({
        "filename": sample.path.name,
        "extracted_text": pytesseract.image_to_string(rgb).strip(),
    })

tesseract_df = pd.DataFrame(tesseract_rows, columns=["filename", "extracted_text"])
tesseract_df

,filename,extracted_text
0,001.jpg,"332 MM @ +\n\nTransfer successful!\n\nSaturday, May 23 2026; 03:32:11 pM (Gn\nConfirmation No. 1779521526959\nTransaction Ref, No. 464438\n\nSent via BP\n\nTransfer to\nGCash/G-Xchange\nJOA C. (QR Code)\nXXOOOOAKXXXXCWU,\n\nTransfer amount\nPHP 300.00\n\nee\nPHP 10.00\n\n/ Hide other details\n\nTransfer from\nSAVINGS ACCOUNT\n\nGo to My Accounts"
1,002.jpg,463 Qee00+6567\n\nAmount\n\nRef No. 8041 012\n628744\n\n< %e\nsar}\nae Add to\n\nShare\n\nry\n\nBi\nJ ¢ } Shot on Y11\n“a? VivoAl camera 2026.05.20 19:45\neye
2,003.jpg,"Transferred\n\nP300.00\n\nInstant\n\no JOR Ce\nScoarnninces Be\nG-Xchange, Inc (GCash)\n\nDERELLE M******\n. +1185\n\nGoTyme Bank\n\n300.00\n\nAmount\n\nTotal 300.00\n\n302011\n\n179260831040302011\n\nReference No\n\n31 Aug 2026 at 12:03 PM"


## 3 | PaddleOCR

In [9]:
# Verify that PaddleOCR is installed successfully
import paddleocr
print(f"PaddleOCR version: {paddleocr.__version__}")

# If you use the local paddle_static inference engine, you can further verify PaddlePaddle and GPU availability
import paddle
print(f"Paddle version: {paddle.__version__}")
print(f"GPU available: {paddle.is_compiled_with_cuda()}")
print(f"GPU count: {paddle.device.cuda.device_count()}")

PaddleOCR version: 3.7.0
Paddle version: 3.3.1
GPU available: True
GPU count: 1


In [11]:
# Use PaddlePaddle for inference
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    engine="paddle",
)

for sample in samples:
    result = ocr.predict(sample.img)
    for res in result:
        res.print()
        res.save_to_img("output")
        res.save_to_json("output")

Creating model: ('PP-OCRv6_medium_det', None, 'paddle')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jmtcabili/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, 'paddle')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jmtcabili/.paddlex/official_models/PP-OCRv6_medium_rec`.
{'res': {'input_path': None, 'page_index': None, 'model_settings': {'use_doc_preprocessor': False, 'use_textline_orientation': False}, 'dt_polys': array([[[ 396,   21],
        ...,
        [ 407,   56]],

       ...,

       [[ 332, 1109],
        ...,
        [ 332, 1139]]], shape=(62, 4, 2), dtype=int16), 'text_det_params': {'limit_side_len': 64, 'limit_type': 'min', 'thresh': 0.3, 'max_side_limit': 4000, 'box_thresh': 0.6, 'unclip_ratio': 1.5}, 'text_type': 'general', 'textline_orientation_angles': array([-1, ..., -1], shape=(62,)), 'text_rec_score_t

# 4 | VLMs (Qwen, GLM, )

In [7]:
from ollama import chat
from ollama import ChatResponse

In [10]:
for sample in samples: 
    print(sample.path)

images/train/001.jpg
images/train/002.jpg
images/train/003.jpg


In [11]:
from pydantic import BaseModel, Field


class Transaction(BaseModel):
    amount: float | None = Field(None, description="Transfer amount as a plain number, excluding fees and totals")
    date: str | None = Field(None, description="Transaction date in ISO format YYYY-MM-DD")
    provider: str | None = Field(None, description="Bank or e-wallet the money was sent from, e.g. BPI, GCash, GoTyme")


SYSTEM_PROMPT = """You extract data from screenshots of Philippine bank / e-wallet transfer receipts.

Return a single JSON object with exactly these fields:
- amount: the amount transferred, as a number with no currency symbol or commas (e.g. 300.00). Use the transfer amount, not the fee and not the total including fee.
- date: the transaction date as YYYY-MM-DD. Ignore the time of day.
- provider: the bank or e-wallet app the money was sent FROM (e.g. "BPI", "GCash", "GoTyme"). Look for phrases like "Sent via" or "From". Do not use the recipient's bank.

Rules:
- Use only information visible in the image. Do not guess.
- If a field is not visible or you are unsure, use null.
- Output JSON only, no explanations."""

In [42]:
STRUCT_MODEL = ["qwen3-vl:2b", "glm-ocr:q8_0", "glm-ocr:latest"]

structured_rows = []
for s in samples:
    r = chat(
        model=STRUCT_MODEL[2],
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Extract the transaction details.", "images": [str(REPO_ROOT / s.path)]},
        ],
        format=Transaction.model_json_schema(),  
        options={"temperature": 0, "num_ctx": 4096}, 
    )
    tx = Transaction.model_validate_json(r.message.content)
    structured_rows.append({"filename": s.path.name, **tx.model_dump()})

structured_df = pd.DataFrame(structured_rows)
structured_df

,filename,amount,date,provider
0,001.jpg,300.0,2026-05-23,BPI
1,002.jpg,300.0,"May 20, 2026",GCash
2,003.jpg,300.0,31 Aug 2026,BPI


In [17]:
import statistics
import time

from ollama import chat

TIMING_MODELS = ["qwen3-vl:2b", "glm-ocr:q8_0", "glm-ocr:latest"]
N_RUNS = 5


def structured_pass(model):
    structured_rows = []
    for s in samples:
        r = chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "Extract the transaction details.", "images": [str(REPO_ROOT / s.path)]},
            ],
            format=Transaction.model_json_schema(),
            options={"temperature": 0, "num_ctx": 2048},
        )
        tx = Transaction.model_validate_json(r.message.content)
        structured_rows.append({"filename": s.path.name, **tx.model_dump()})
    return structured_rows


timing_runs = []   # one row per (model, run)
timing_summary = []

for model in TIMING_MODELS:
    try:
        structured_pass(model)  # warm-up: loads the model, not timed
    except Exception as e:
        timing_summary.append({"model": model, "ok_runs": 0, "mean_total_s": None,
                               "stdev_total_s": None, "mean_per_image_s": None,
                               "note": f"warm-up failed: {type(e).__name__}: {str(e)[:120]}"})
        print(e)
        continue

    run_totals = []
    outputs=[]
    for run in range(1, N_RUNS + 1):
        try:
            t0 = time.perf_counter()
            output = structured_pass(model)
            elapsed = time.perf_counter() - t0
        except Exception as e:
            timing_runs.append({"model": model, "run": run, "total_s": None,
                                "per_image_s": None, "error": f"{type(e).__name__}: {str(e)[:120]}"})
            continue

        run_totals.append(elapsed)
        timing_runs.append({"model": model, "run": run, "total_s": round(elapsed, 3),
                            "per_image_s": round(elapsed / len(samples), 3), "error": ""})

    timing_summary.append({
        "model": model,
        "ok_runs": len(run_totals),
        "mean_total_s": round(statistics.mean(run_totals), 3) if run_totals else None,
        "stdev_total_s": round(statistics.stdev(run_totals), 3) if len(run_totals) > 1 else None,
        "mean_per_image_s": round(statistics.mean(run_totals) / len(samples), 3) if run_totals else None,
        "note": "",
    })

timing_runs_df = pd.DataFrame(timing_runs)
timing_summary_df = pd.DataFrame(timing_summary)
print(f"Images per run: {len(samples)}, runs per model: {N_RUNS} (plus one untimed warm-up)")
display(timing_summary_df)
timing_runs_df


an error was encountered while running the model: unexpected EOF (status code: 500)
an error was encountered while running the model: unexpected EOF (status code: 500)
Images per run: 3, runs per model: 5 (plus one untimed warm-up)


,model,ok_runs,mean_total_s,stdev_total_s,mean_per_image_s,note
0,qwen3-vl:2b,5,17.829,0.047,5.943,
1,glm-ocr:q8_0,0,NaN,NaN,NaN,warm-up failed: ResponseError: an error was en...
2,glm-ocr:latest,0,NaN,NaN,NaN,warm-up failed: ResponseError: an error was en...


,model,run,total_s,per_image_s,error
0,qwen3-vl:2b,1,17.755,5.918,
1,qwen3-vl:2b,2,17.813,5.938,
2,qwen3-vl:2b,3,17.864,5.955,
3,qwen3-vl:2b,4,17.865,5.955,
4,qwen3-vl:2b,5,17.849,5.950,
